# Сравнение моделей регрессии и подбор гиперпараметров

Ноутбук использует уже подготовленный в проекте набор `model_df`. Повторный EDA не выполняется: данные очищаются штатными функциями приложения, после чего сравниваются пять регрессионных алгоритмов и подбираются их гиперпараметры.

Сравнение проводится на отложенной тестовой выборке. Подбор параметров выполняется кросс-валидацией только на обучающей части. Целевая переменная — логарифм цены квадратного метра; итоговый прогноз переводится обратно в цену квартиры.

jupyter notebook

## 1. Библиотеки и пути к данным

Ячейка находит корень проекта по исходному CSV, подключает функции приложения для подготовки данных и задаёт путь, по которому будет сохранён файл моделей.


In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge
from catboost import CatBoostRegressor

ROOT = Path.cwd()
CSV_NAME = "СДЕЛКИ_Мос_2023-2026.csv"
while not (ROOT / CSV_NAME).exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / CSV_NAME).exists():
    raise FileNotFoundError("Source CSV was not found in the project root")
sys.path.insert(0, str(ROOT / "web_app"))
import model as app_model

DATA_PATH = ROOT / CSV_NAME
BUNDLE_PATH = ROOT / "Prognosis_model.joblib"
RANDOM_STATE = 42
CAT_COLS = ["region", "district", "street", "wall_material_code"]
NUM_COLS = ["year_build", "floor", "area"]
FEATURE_COLS = CAT_COLS + NUM_COLS + ["period_year"]
FINAL_BASE_YEAR = 2026
print("Project root:", ROOT)
print("Data file:", DATA_PATH)
print("Bundle path:", BUNDLE_PATH)

Project root: C:\Users\AnnaA\Desktop\Prognosis_system
Data file: C:\Users\AnnaA\Desktop\Prognosis_system\СДЕЛКИ_Мос_2023-2026.csv
Bundle path: C:\Users\AnnaA\Desktop\Prognosis_system\Prognosis_model.joblib


## 2. Настройки эксперимента

Используется ускоренный подбор: 3 кандидата × 3 фолда = 9 CV-обучений на модель. Для CV берётся не более 20 000 строк обучающей выборки. После выбора параметров каждая конфигурация обучается ровно один раз на полном `X_train`; это обучение одновременно используется для расчёта тестовых метрик и измерения `Time, s`. Параллелизм ограничен двумя процессами для снижения нагрузки на оперативную память.

In [2]:
# Быстрые настройки эксперимента
TEST_SIZE = 0.20
CV_SPLITS = 3                 # было 5
N_ITER_SEARCH = 3             # 9 CV fit на модель вместо 60
MAX_SEARCH_ROWS = 20000       # CV-подбор на подвыборке; финальное fit — на полном train
RANDOM_STATE = 42
N_JOBS = 2                    # ограничиваем параллелизм, чтобы не перегружать RAM

print({
    "test_size": TEST_SIZE,
    "cv_splits": CV_SPLITS,
    "iterations_per_model": N_ITER_SEARCH,
    "fits_per_model_during_search": CV_SPLITS * N_ITER_SEARCH,
    "max_rows_for_search": MAX_SEARCH_ROWS,
    "search_parallel_jobs": N_JOBS,
    "random_state": RANDOM_STATE,
})

{'test_size': 0.2, 'cv_splits': 3, 'iterations_per_model': 3, 'fits_per_model_during_search': 9, 'max_rows_for_search': 20000, 'search_parallel_jobs': 2, 'random_state': 42}


## 3. Подготовка обучающей выборки

Применяются те же очистка и фильтры, что и в приложении. Повторный EDA не выполняется. Финальное обучение победившей модели проводится на всей очищенной выборке, включая 2026 год; для сравнения моделей используется отдельная отложенная тестовая выборка.

In [3]:
df_raw = app_model.read_csv_robust(DATA_PATH)
df_raw.columns = df_raw.columns.astype(str).str.strip().str.lower()
clean_df = app_model.clean_data(df_raw)
# Финальный набор готовится приложением по её производственной схеме.
model_df = app_model._prepare_model_data(clean_df)
if model_df.empty or model_df["period_year"].max() < FINAL_BASE_YEAR:
    raise ValueError("В очищенной выборке должны присутствовать сделки за 2026 год")
print(f"Строк финального обучения: {len(model_df):,}")
display(model_df.groupby("period_year").agg(deals=("price", "size"), median_price=("price", "median")))

Строк финального обучения: 60,289


,deals,median_price
period_year,,
2023,2999,23538610.0
2024,18148,23603680.0
2025,23702,23500000.0
2026,15440,22000000.0


## 4. Подготовка признаков и алгоритмов

Категориальные признаки кодируются отдельно для разных семейств моделей:
- Ridge Regression — One-Hot Encoding;
- Random Forest и Extra Trees — One-Hot Encoding;
- HistGradientBoosting — Ordinal Encoding с указанием категориальных столбцов;
- CatBoost — нативная обработка категориальных признаков.

Во всех случаях прогнозируется логарифм цены квадратного метра. При расчёте итоговых метрик прогноз умножается на площадь квартиры.

Для CatBoost используется sklearn-совместимая обёртка с неизменяемым tuple для `cat_features`. Это предотвращает ошибку `Cannot clone object ... modifies parameter cat_features` при работе `RandomizedSearchCV`.

In [4]:
import time
import warnings
from IPython.display import display, HTML
from scipy.stats import randint, loguniform
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore", category=FutureWarning)


class CloneSafeCatBoostRegressor(BaseEstimator, RegressorMixin):
    """Scikit-learn-compatible CatBoost wrapper; keeps cat_features immutable for clone()."""

    def __init__(
        self,
        iterations=200,
        depth=6,
        learning_rate=0.1,
        l2_leaf_reg=3,
        loss_function="MAE",
        random_seed=42,
        thread_count=4,
        allow_writing_files=False,
        cat_features=("region", "district", "street", "wall_material_code"),
        verbose=False,
    ):
        # Do not transform constructor arguments here: sklearn.clone checks identity.
        self.iterations = iterations
        self.depth = depth
        self.learning_rate = learning_rate
        self.l2_leaf_reg = l2_leaf_reg
        self.loss_function = loss_function
        self.random_seed = random_seed
        self.thread_count = thread_count
        self.allow_writing_files = allow_writing_files
        self.cat_features = cat_features
        self.verbose = verbose

    def fit(self, X, y):
        self.model_ = CatBoostRegressor(
            iterations=self.iterations,
            depth=self.depth,
            learning_rate=self.learning_rate,
            l2_leaf_reg=self.l2_leaf_reg,
            loss_function=self.loss_function,
            random_seed=self.random_seed,
            thread_count=self.thread_count,
            allow_writing_files=self.allow_writing_files,
            cat_features=list(self.cat_features),
            verbose=self.verbose,
        )
        self.model_.fit(X, y)
        self.n_features_in_ = X.shape[1]
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def predict(self, X):
        return self.model_.predict(X)

# X содержит исходные признаки; y_log — логарифм цены квадратного метра.
X = model_df[FEATURE_COLS].copy()
y_price = model_df["price"].to_numpy(dtype=float)
areas = model_df["area"].to_numpy(dtype=float)
y_log = np.log(y_price / areas)

# Разделение с сохранением пропорций годов в train и test.
X_train, X_test, y_train_log, y_test_log, y_train_price, y_test_price, area_train, area_test = train_test_split(
    X, y_log, y_price, areas,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=X["period_year"]
)

cat_features = CAT_COLS
numeric_features = NUM_COLS + ["period_year"]

def make_onehot_preprocessor(scale_numeric=False):
    """Преобразование категориальных и числовых признаков для линейной модели и деревьев."""
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", min_frequency=5)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore")
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer(
        transformers=[
            ("categorical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", encoder)
            ]), cat_features),
            ("numeric", Pipeline(numeric_steps), numeric_features),
        ],
        remainder="drop"
    )

def make_ordinal_preprocessor():
    # OrdinalEncoder ограничивает число категорий до 200.
    # HistGradientBoosting ниже использует max_bins=255, поэтому 146 категорий допустимы.
    ordinal_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        max_categories=200
    )
    return ColumnTransformer(
        transformers=[
            ("categorical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", ordinal_encoder)
            ]), cat_features),
            ("numeric", SimpleImputer(strategy="median"), numeric_features),
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )

# После OrdinalEncoder первые четыре столбца — категориальные.
hgb_categorical_mask = [True] * len(cat_features) + [False] * len(numeric_features)

cv = KFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)
scoring = "neg_mean_absolute_error"

search_spaces = {
    "Ridge Regression": {
        "estimator": Pipeline([
            ("preprocessor", make_onehot_preprocessor(scale_numeric=True)),
            ("model", Ridge())
        ]),
        "params": {
            "model__alpha": loguniform(0.1, 1000),
        },
    },
    "RandomForestRegressor": {
        "estimator": Pipeline([
            ("preprocessor", make_onehot_preprocessor()),
            ("model", RandomForestRegressor(
                random_state=RANDOM_STATE, n_jobs=1
            ))
        ]),
        "params": {
            "model__n_estimators": randint(40, 81),
            "model__max_depth": [12, 18],
            "model__min_samples_leaf": randint(3, 9),
            "model__max_features": [0.5, 0.8],
        },
    },
    "ExtraTreesRegressor": {
        "estimator": Pipeline([
            ("preprocessor", make_onehot_preprocessor()),
            ("model", ExtraTreesRegressor(
                random_state=RANDOM_STATE, n_jobs=1
            ))
        ]),
        "params": {
            "model__n_estimators": randint(40, 81),
            "model__max_depth": [12, 18],
            "model__min_samples_leaf": randint(3, 9),
            "model__max_features": [0.5, 0.8],
        },
    },
    "HistGradientBoostingRegressor": {
        "estimator": Pipeline([
            ("preprocessor", make_ordinal_preprocessor()),
            ("model", HistGradientBoostingRegressor(
                categorical_features=hgb_categorical_mask,
                max_bins=255,          # фиксировано на 255: устраняет ошибку cardinality 146 > 128
                max_iter=120,
                early_stopping=True,
                random_state=RANDOM_STATE
            ))
        ]),
        "params": {
            "model__learning_rate": [0.05, 0.1],
            "model__max_iter": [80, 120, 160],
            "model__max_leaf_nodes": [15, 31],
            "model__min_samples_leaf": [20, 40],
        },
    },
    "CatBoostRegressor": {
        "estimator": CloneSafeCatBoostRegressor(
            loss_function="MAE",
            verbose=False,
            random_seed=RANDOM_STATE,
            thread_count=4,
            allow_writing_files=False,
            cat_features=tuple(CAT_COLS)
        ),
        "params": {
            "iterations": randint(150, 301),
            "depth": [5, 6, 7],
            "learning_rate": [0.05, 0.1],
            "l2_leaf_reg": [3, 8],
        },
    },
}

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Категориальные признаки:", cat_features)
print("Числовые признаки:", numeric_features)
print(f"Подбор: {N_ITER_SEARCH} кандидата × {CV_SPLITS} фолда = {N_ITER_SEARCH * CV_SPLITS} обучений на модель.")


Train: (48231, 8) | Test: (12058, 8)
Категориальные признаки: ['region', 'district', 'street', 'wall_material_code']
Числовые признаки: ['year_build', 'floor', 'area', 'period_year']
Подбор: 3 кандидата × 3 фолда = 9 обучений на модель.


## 5. Подбор гиперпараметров и сравнение моделей

Для каждой модели выполняется ускоренный `RandomizedSearchCV`: 3 кандидата и 3 фолда на подвыборке до 20 000 строк (9 CV-обучений на алгоритм). После подбора лучшая конфигурация обучается ровно один раз на полном `X_train`. Время этого вызова `fit()` записывается в `Time, s`; отдельные повторные обучения не выполняются.

In [5]:
search_results = []
best_estimators = {}
best_hyperparameters = {}
single_fit_times = {}

for algorithm, config in search_spaces.items():
    print(f"\nПодбор: {algorithm}")

    # Для ускорения подбор выполняется на ограниченной подвыборке train.
    # Итоговая модель и тестовые метрики далее рассчитываются на полном train/test.
    search_n = min(MAX_SEARCH_ROWS, len(X_train))
    X_search = X_train.sample(n=search_n, random_state=RANDOM_STATE)
    y_train_series = pd.Series(y_train_log, index=X_train.index)
    y_search_log = y_train_series.loc[X_search.index].to_numpy()

    search = RandomizedSearchCV(
        estimator=config["estimator"],
        param_distributions=config["params"],
        n_iter=N_ITER_SEARCH,
        scoring=scoring,
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        pre_dispatch=N_JOBS,
        refit=False,
        verbose=1,
        error_score="raise",
        return_train_score=False,
    )

    # Подбор гиперпараметров на подвыборке. refit=False исключает лишнее переобучение.
    search.fit(X_search, y_search_log)

    best_hyperparameters[algorithm] = search.best_params_

    # Обучаем найденную конфигурацию ровно один раз на полном X_train.
    # Это же время записываем в Time, s; отдельные повторные fit не выполняются.
    best_model = clone(config["estimator"]).set_params(**search.best_params_)
    started = time.perf_counter()
    best_model.fit(X_train, y_train_log)
    mean_single_fit_time = float(time.perf_counter() - started)

    best_estimators[algorithm] = best_model
    single_fit_times[algorithm] = mean_single_fit_time

    # Оценка лучшей модели на общей отложенной тестовой выборке.
    predicted_log = best_model.predict(X_test)
    predicted_price = np.exp(predicted_log) * area_test

    mae_mln = mean_absolute_error(y_test_price, predicted_price) / 1e6
    rmse_mln = np.sqrt(mean_squared_error(y_test_price, predicted_price)) / 1e6
    mape = np.mean(
        np.abs(y_test_price - predicted_price) /
        np.maximum(y_test_price, 1.0)
    ) * 100
    r2 = r2_score(y_test_price, predicted_price)
    cv_mae_log = -float(search.best_score_)

    search_results.append({
        "Algorithm": algorithm,
        "MAE,\nmillion RUB": mae_mln,
        "RMSE,\nmillion RUB": rmse_mln,
        "MAPE,\n%": mape,
        "R²": r2,
        "CV\nMAE (log)": cv_mae_log,
        "Time,\ns": mean_single_fit_time,
        "Accuracy, %": 100 - mape,
    })

    print(f"CV MAE (log): {cv_mae_log:.6f}")
    print(f"Лучшие гиперпараметры: {search.best_params_}")
    print(
        f"Время одного обучения (один вызов fit): {mean_single_fit_time:.3f} с"
    )

# Не допускаем отображения пустой/частичной таблицы как будто эксперимент завершён.
expected_algorithms = [
    "Ridge Regression",
    "RandomForestRegressor",
    "ExtraTreesRegressor",
    "HistGradientBoostingRegressor",
    "CatBoostRegressor",
]
if set(best_estimators) != set(expected_algorithms) or len(search_results) != len(expected_algorithms):
    raise RuntimeError(
        "Сравнение не завершено: должны быть обучены все пять моделей. "
        f"Получены результаты: {list(best_estimators)}"
    )

# Итоговая таблица: ровно 8 столбцов, в заданном порядке.
comparison_table = pd.DataFrame(search_results, columns=[
    "Algorithm",
    "MAE,\nmillion RUB",
    "RMSE,\nmillion RUB",
    "MAPE,\n%",
    "R²",
    "CV\nMAE (log)",
    "Time,\ns",
    "Accuracy, %",
]).sort_values("CV\nMAE (log)", ascending=True).reset_index(drop=True)

# Визуальные переносы строк в заголовках соответствуют требуемому макету таблицы.
display_table = comparison_table.round(4).copy()
display_table.columns = [
    "Algorithm",
    "MAE,<br><br>million RUB",
    "RMSE,<br><br>million RUB",
    "MAPE,<br><br>%",
    "R²",
    "CV<br><br>MAE (log)",
    "Time,<br><br>s",
    "Accuracy, %",
]
print("ИТОГОВОЕ СРАВНЕНИЕ ПЯТИ МОДЕЛЕЙ")
print("Time, s = среднее время одного вызова fit() для лучшей конфигурации модели.")
expected_display_columns = [
    "Algorithm",
    "MAE,<br><br>million RUB",
    "RMSE,<br><br>million RUB",
    "MAPE,<br><br>%",
    "R²",
    "CV<br><br>MAE (log)",
    "Time,<br><br>s",
    "Accuracy, %",
]
if list(display_table.columns) != expected_display_columns:
    raise AssertionError(
        "Нарушены названия или порядок столбцов итоговой таблицы: "
        f"{list(display_table.columns)}"
    )
if display_table.shape != (5, 8):
    raise AssertionError(
        f"Ожидалась таблица 5×8, получена {display_table.shape}"
    )

html_table = display_table.to_html(index=False, escape=False, border=0, classes="model-comparison")
display(HTML(
    "<style>"
    ".model-comparison {border-collapse:collapse; width:100%; font-size:13px;}"
    ".model-comparison th, .model-comparison td {border:1px solid #d0d7de; padding:8px 10px; text-align:center;}"
    ".model-comparison th {font-weight:700; background:#f3f4f6; white-space:normal;}"
    ".model-comparison td:first-child, .model-comparison th:first-child {text-align:left;}"
    "</style>" + html_table
))


Подбор: Ridge Regression
Fitting 3 folds for each of 3 candidates, totalling 9 fits
CV MAE (log): 0.228962
Лучшие гиперпараметры: {'model__alpha': np.float64(3.14891164795686)}
Время одного обучения (один вызов fit): 0.821 с

Подбор: RandomForestRegressor
Fitting 3 folds for each of 3 candidates, totalling 9 fits
CV MAE (log): 0.221785
Лучшие гиперпараметры: {'model__max_depth': 12, 'model__max_features': 0.8, 'model__min_samples_leaf': 7, 'model__n_estimators': 54}
Время одного обучения (один вызов fit): 25.289 с

Подбор: ExtraTreesRegressor
Fitting 3 folds for each of 3 candidates, totalling 9 fits
CV MAE (log): 0.225539
Лучшие гиперпараметры: {'model__max_depth': 12, 'model__max_features': 0.8, 'model__min_samples_leaf': 5, 'model__n_estimators': 62}
Время одного обучения (один вызов fit): 23.304 с

Подбор: HistGradientBoostingRegressor
Fitting 3 folds for each of 3 candidates, totalling 9 fits
CV MAE (log): 0.218726
Лучшие гиперпараметры: {'model__min_samples_leaf': 20, 'model__ma

Algorithm,"MAE,million RUB","RMSE,million RUB","MAPE,%",R²,CVMAE (log),"Time,s","Accuracy, %"
CatBoostRegressor,6.0297,10.3507,18.2835,0.4652,0.2078,18.3115,81.7165
HistGradientBoostingRegressor,6.2763,9.7700,20.8134,0.5236,0.2187,0.8785,79.1866
RandomForestRegressor,6.4833,9.9784,21.5722,0.5030,0.2218,25.2892,78.4278
ExtraTreesRegressor,6.5308,10.0195,21.8405,0.4989,0.2255,23.3038,78.1595
Ridge Regression,6.6126,10.1380,22.1571,0.4870,0.2290,0.8209,77.8429


## 6. Победившая модель и лучшие гиперпараметры

Победитель определяется по минимальному значению `CV MAE (log)`, то есть по критерию, который использовался непосредственно при подборе гиперпараметров. Ниже выводятся название алгоритма, все найденные лучшие параметры и его итоговые метрики на тестовой выборке.

In [6]:
if "comparison_table" not in globals() or comparison_table.empty or len(comparison_table) != 5:
    raise RuntimeError("Сначала полностью выполните ячейку подбора всех пяти моделей.")

winner_name = comparison_table.loc[0, "Algorithm"]
winner_params = best_hyperparameters[winner_name]
winner_model = best_estimators[winner_name]
winner_row = comparison_table.iloc[0]

print("=" * 72)
print("ПОБЕДИВШАЯ МОДЕЛЬ:", winner_name)
print("КРИТЕРИЙ ПОБЕДЫ: минимальная CV MAE (log)")
cv_mae_value = winner_row["CV\nMAE (log)"]
print(f"CV MAE (log): {cv_mae_value:.6f}")
print("\nВСЕ ЛУЧШИЕ ГИПЕРПАРАМЕТРЫ ПОБЕДИВШЕЙ МОДЕЛИ:")
print(json.dumps(winner_params, ensure_ascii=False, indent=4, default=str))
print(f"Среднее время одного обучения: {single_fit_times[winner_name]:.3f} с")
print("\nМЕТРИКИ ПОБЕДИТЕЛЯ НА ОТЛОЖЕННОЙ ТЕСТОВОЙ ВЫБОРКЕ:")
for column in [
    "MAE,\nmillion RUB", "RMSE,\nmillion RUB", "MAPE,\n%", "R²", "Accuracy, %"
]:
    print(f"{column.replace(chr(10), ' ')}: {winner_row[column]:.4f}")
print("=" * 72)

ПОБЕДИВШАЯ МОДЕЛЬ: CatBoostRegressor
КРИТЕРИЙ ПОБЕДЫ: минимальная CV MAE (log)
CV MAE (log): 0.207768

ВСЕ ЛУЧШИЕ ГИПЕРПАРАМЕТРЫ ПОБЕДИВШЕЙ МОДЕЛИ:
{
    "depth": 7,
    "iterations": 224,
    "l2_leaf_reg": 3,
    "learning_rate": 0.1
}
Среднее время одного обучения: 18.311 с

МЕТРИКИ ПОБЕДИТЕЛЯ НА ОТЛОЖЕННОЙ ТЕСТОВОЙ ВЫБОРКЕ:
MAE, million RUB: 6.0297
RMSE, million RUB: 10.3507
MAPE, %: 18.2835
R²: 0.4652
Accuracy, %: 81.7165


## 7. Финальное обучение победившей модели на всех данных

После завершения сравнения победивший алгоритм повторно обучается на полном подготовленном наборе `model_df` с найденными гиперпараметрами. Тестовая оценка выше остаётся независимой оценкой качества; финальная модель предназначена для последующего использования.

In [7]:
X_full = model_df[FEATURE_COLS].copy()
y_full_log = np.log(model_df["price"].to_numpy(dtype=float) / model_df["area"].to_numpy(dtype=float))

# Создаём экземпляр той же модели с найденными параметрами.
if winner_name == "CatBoostRegressor":
    final_model = CatBoostRegressor(
        loss_function="MAE",
        verbose=False,
        random_seed=RANDOM_STATE,
        thread_count=4,
        allow_writing_files=False,
        cat_features=tuple(CAT_COLS),
        **winner_params
    )
else:
    # Для Pipeline параметры имеют префиксы этапов; best_estimator_ уже содержит
    # точную конфигурацию и обучается заново на полном наборе.
    final_model = clone(winner_model)

final_model.fit(X_full, y_full_log)

print("Финальная модель обучена на всех строках model_df.")
print("Алгоритм:", winner_name)
print("Количество объектов:", len(X_full))
print("Гиперпараметры:")
print(json.dumps(winner_params, ensure_ascii=False, indent=4, default=str))

Финальная модель обучена на всех строках model_df.
Алгоритм: CatBoostRegressor
Количество объектов: 60289
Гиперпараметры:
{
    "depth": 7,
    "iterations": 224,
    "l2_leaf_reg": 3,
    "learning_rate": 0.1
}


## 8. Сводка эксперимента

В итоговой таблице представлены все пять алгоритмов, их ошибки на общей тестовой выборке, CV MAE (log) и время одного обучения лучшей конфигурации на полном `X_train`. Для ускорения CV-подбор ограничен 20 000 строками и 3 фолдами × 3 кандидатами на модель. Само итоговое обучение каждой выбранной конфигурации и оценка на общей тестовой выборке выполняются на полном train/test.

In [8]:
# Финальный машинно-читаемый вывод победителя и всех его лучших гиперпараметров.
if "winner_name" not in globals() or "best_hyperparameters" not in globals():
    raise RuntimeError("Сначала выполните подбор всех пяти моделей и ячейку определения победителя.")

print("ПОБЕДИТЕЛЬ ЭКСПЕРИМЕНТА:", winner_name)
print("КРИТЕРИЙ ВЫБОРА: минимальная CV MAE (log)")
print("ВСЕ ЛУЧШИЕ ПОДОБРАННЫЕ ГИПЕРПАРАМЕТРЫ ПОБЕДИТЕЛЯ:")
BEST_PARAMS = best_hyperparameters[winner_name]
print(json.dumps(BEST_PARAMS, ensure_ascii=False, indent=4, default=str))

ПОБЕДИТЕЛЬ ЭКСПЕРИМЕНТА: CatBoostRegressor
КРИТЕРИЙ ВЫБОРА: минимальная CV MAE (log)
ВСЕ ЛУЧШИЕ ПОДОБРАННЫЕ ГИПЕРПАРАМЕТРЫ ПОБЕДИТЕЛЯ:
{
    "depth": 7,
    "iterations": 224,
    "l2_leaf_reg": 3,
    "learning_rate": 0.1
}
